In [2]:
# creating a simple chroma collection with 5 documents
import chromadb
from llama_index.embeddings.ollama import OllamaEmbedding


embed_model = OllamaEmbedding(model_name="embeddinggemma")
PERSIST_DIR = "chromadb_bug"
COLLECTION_NAME = "vector_store_test6"
chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

for i in range(5):
    id = f"doc_{i}"
    doc = f"This is document {i}"
    embedding = embed_model.get_text_embedding(doc)
    collection.add(
        documents=[doc],
        embeddings=[embedding],
        ids=[id],
    )

In [3]:
# retrieving documents with similarity search with llamaindex module
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings
Settings.embed_model = embed_model
top_k = 3


vector_store = ChromaVectorStore(chroma_collection=collection)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
retriever = index.as_retriever(similarity_top_k=top_k)

query_text = "get document 1"
nodes = retriever.retrieve(query_text)

print(f"\nTop {top_k} similar documents to '{query_text}':\n")
for i, node in enumerate(nodes, 1):
    print(f"Document {i}:")
    print(f"Score: {node.score:.4f}")
    print(f"Content: {node.text}\n")




Top 3 similar documents to 'get document 1':

Document 1:
Score: 0.0000
Content: This is document 1

Document 2:
Score: 0.0000
Content: This is document 0

Document 3:
Score: 0.0000
Content: This is document 4



In [4]:
# retrieving nodes with chromadb similarity search
query_embedding = embed_model.get_text_embedding(query_text)
results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
print(f"\nTop {top_k} similar documents to '{query_text}' using chromadb query:\n")
for i, (doc, score) in enumerate(zip(results['documents'][0], results['distances'][0]), 1):
    print(f"Document {i}:")
    print(f"Score: {score:.4f}")
    print(f"Content: {doc}\n")


Top 3 similar documents to 'get document 1' using chromadb query:

Document 1:
Score: 88770.7188
Content: This is document 1

Document 2:
Score: 98965.4375
Content: This is document 0

Document 3:
Score: 123902.4062
Content: This is document 4



In [7]:
import math
from pprint import pprint

# query_embedding già ottenuta come lista di float:
query_embedding = embed_model.get_text_embedding("get document 1")

# Force include esplicito per vedere tutto
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "distances", "embeddings"]
)

print("RESULT KEYS:", list(results.keys()))
pprint(results)

# controlli più dettagliati su 'distances'
distances = results.get("distances", None)
print("distances raw:", distances)

if distances is None:
    print(">>> distances è None (assenza del campo).")
elif len(distances) == 0 or distances[0] is None:
    print(">>> distances è vuoto o contiene None:", distances)
else:
    for i, d in enumerate(distances[0]):
        print(f"idx {i}: type={type(d)}, value={d}, finite={isinstance(d, float) and math.isfinite(d)}")


RESULT KEYS: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']
{'data': None,
 'distances': [[88770.71875, 98965.4375, 123902.40625]],
 'documents': [['This is document 1',
                'This is document 0',
                'This is document 4']],
 'embeddings': [array([[-103.57275391,    9.29625988,   13.12675667, ...,  -26.08492851,
          -2.26202536,  -18.39478683],
       [-106.86103821,    8.84418201,    3.69188309, ...,  -11.02489471,
         -19.65815544,   -7.33786201],
       [-107.22244263,   19.08135796,    6.7651825 , ...,  -24.55737495,
         -20.48596954,   -2.49409294]])],
 'ids': [['doc_1', 'doc_0', 'doc_4']],
 'included': ['documents', 'distances', 'embeddings'],
 'metadatas': None,
 'uris': None}
distances raw: [[88770.71875, 98965.4375, 123902.40625]]
idx 0: type=<class 'float'>, value=88770.71875, finite=True
idx 1: type=<class 'float'>, value=98965.4375, finite=True
idx 2: type=<class 'float'>, value=123902.40625, f

In [12]:
math.exp(-distances[0][0])

0.0